### Load Modules

In [7]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path(os.getcwd()).resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from torchsummary import summary
from PIL import Image

import torch
import torch.nn.functional as F
import torch.optim as optim
import torch.nn as nn

import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Subset, random_split
from torch.optim.lr_scheduler import StepLR

import wandb
from torchsummary import summary

### Mount to Google drive

In [8]:
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)
    print("Drive mounted successfully!")
else:
    print("Drive already mounted.")

Drive already mounted.


### Clone git and load modules

In [9]:
!git clone https://github.com/gimoonnam/vgg16_practice.git

fatal: destination path 'vgg16_practice' already exists and is not an empty directory.


In [11]:
repo_path = '/content/vgg16_practice'
if repo_path not in sys.path:
  sys.path.insert(0, repo_path)

from load_data import CatDogDataLoadandSave
from vgg16_model import VGG16
from data_classes import TrainingConfig

### Load data and Save dataset as ubyte format


In [19]:
data_path = r'/content/drive/My Drive/Data for Colab Training'
train_data_path = os.path.join(data_path, "cat-and-dog", "training_set")
test_data_path  = os.path.join(data_path, "cat-and-dog", "test_set")

# Load full dataset
dataset_train = CatDogDataLoadandSave(data_dir=train_data_path)

print(f"Original dataset size: {len(dataset_train)}")

# Split into train and validation sets (e.g., 80/20 split)
train_ratio = 0.8
train_size = int(train_ratio * len(dataset_train))
val_size = len(dataset_train) - train_size

train_dataset, val_dataset = random_split(
    dataset_train,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)  # For reproducibility
)

print(f"Training set size: {len(train_dataset)} ({train_ratio*100:.0f}%)")
print(f"Validation set size: {len(val_dataset)} ({(1-train_ratio)*100:.0f}%)")


# create dataclass for training config
config = TrainingConfig(
    batch_size=32,
    num_epochs=20,
    num_train_samples=train_size,  # Set from your dataset
    lr_scheduler_epoch=5
)


lr_scheduler_step_size = config.lr_scheduler_step_size()
print(f"\nnum of steps at every {config.lr_scheduler_epoch} epoch: {lr_scheduler_step_size}")

total_steps = config.total_steps()
print(f"total number of steps: {total_steps}")


train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")


Original dataset size: 8007
Training set size: 6405 (80%)
Validation set size: 1602 (20%)

num of steps at every 5 epoch: 1000
total number of steps: 4000

Train batches: 201
Validation batches: 51


### Build VGG16 architecture

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VGG16(3, 2).to(device)

print(device)

summary(model, (3, 224, 224))


cuda
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 224, 224]           1,792
              ReLU-2         [-1, 64, 224, 224]               0
            Conv2d-3         [-1, 64, 224, 224]          36,928
              ReLU-4         [-1, 64, 224, 224]               0
         MaxPool2d-5         [-1, 64, 112, 112]               0
            N_conv-6         [-1, 64, 112, 112]               0
            Conv2d-7        [-1, 128, 112, 112]          73,856
              ReLU-8        [-1, 128, 112, 112]               0
            Conv2d-9        [-1, 128, 112, 112]         147,584
             ReLU-10        [-1, 128, 112, 112]               0
        MaxPool2d-11          [-1, 128, 56, 56]               0
           N_conv-12          [-1, 128, 56, 56]               0
           Conv2d-13          [-1, 256, 56, 56]         295,168
             ReLU-14          [-1,

In [20]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
scheduler = StepLR(optimizer, step_size=config.lr_scheduler_step_size, gamma=0.5)

In [ ]:
# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="gimoonnam",
    # Set the wandb project where this run will be logged.
    project="vgg16_practice",
    # Track hyperparameters and,
    name = f"run_with_steplr_{config.num_train_samples}_{config.batch_size}",

    config={
        "learning_rate": config.learning_rate,
        "architecture": "CNN",
        "dataset": "cat-and-dog",
        "epochs": config.num_epochs,
    },
)



model.train()

for epoch in range(config.num_epochs):
    ProgressBar = tqdm(enumerate(train_loader), total=len(train_loader))

    for batch_idx, (inputs, labels) in ProgressBar:

        # Ensure labels are torch.long before moving to device for CrossEntropyLoss
        inputs, labels = inputs.to(device), labels.to(device)
        labels = labels.long()

        optimizer.zero_grad()
        outputs = model(inputs)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()


        # validate
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs_val, labels_val in val_loader:
                inputs_val, labels_val = inputs_val.to(device), labels_val.to(device)
                outputs_val = model(inputs_val)
                loss = criterion(outputs_val, labels_val)
                val_loss += loss.item()
            avg_val_loss = val_loss / len(val_loader)

        # decay lr
        scheduler.step()

        #Update Progress bar
        ProgressBar.set_description(f'Epoch [{epoch+1}]')
        ProgressBar.set_postfix(loss=loss.item())
        ProgressBar.set_postfix(val_loss=avg_val_loss)
        ProgressBar.set_postfix(lr=scheduler.get_last_lr())

        # Log metrics to wandb
        run.log({"loss": loss, "val_loss": avg_val_loss, "lr": scheduler.get_last_lr()})

run.finish()

  0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
from vgg16_model import save_checkpoint, load_checkpoint


checkpoint_path = r'/content/drive/My Drive/checkpoints/cat-and-dog'
save_checkpoint(checkpoint_path, batch_size, num_epochs, model, optimizer, loss)

# pth_file_path = os.path.join(checkpoint_path, 'checkpoint-2026-02-12 09_10_08.pth')
# model, optimizer, epoch, loss = load_checkpoint(pth_file_path, model, optimizer)